# Utilization Estimators with `Scikit-Learn`

The task is to estimate the maximum utilization of a particular cross section under a set of internal forces.

In [1]:
import pandas as pd
from sklearn.model_selection import (
    train_test_split,
    KFold,
    cross_val_score,
    RandomizedSearchCV,
)
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.base import BaseEstimator
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error,
    median_absolute_error,
)
from utils import CANONICAL_SCORE_NAME, INTERNAL_FORCE_COMPONENTS, print_system_info
from utils.ml import canonical_regression_score
import numpy as np
import mlflow
import json
from typing import Optional

print_system_info()

Python version: 3.12.7 (v3.12.7:0b05ead877f, Sep 30 2024, 23:18:00) [Clang 13.0.0 (clang-1300.0.29.30)]
Operating System: Darwin 23.4.0
Platform: macOS-14.4-arm64-arm-64bit
Processor: arm
Machine: arm64
CPU count: 11


In [3]:
config_file_path = "config_rhs.json"
data_file_path = "data_rhs.csv"
mlflow_tracking_uri="sqlite:///mlflow.db"
mlflow_experiment_name=None
task = "utilization_estimation"

In [4]:
with open(config_file_path, "r") as f:
    config: dict = json.load(f)

In [ ]:
# Load section data
section_data = config["section"]

section_type = config["section"]["type"]
print(f"Section type from config: {section_type}")

section_variables = []
for p in section_data["params"].keys():
    if section_data["params"][p]["variable"]:
        section_variables.append(p)

predictor_columns = section_variables + INTERNAL_FORCE_COMPONENTS
target_columns = ["utilization"]

if not mlflow_experiment_name:
    mlflow_experiment_name = f"{task}__{section_type}"

Section type from config: rectangular_hollow_section


## Prepare data

In [6]:
df = pd.read_csv(data_file_path)
df = df.dropna()
assert df.isnull().sum().max() == 0, "DataFrame still contains NaN values after dropping."
print("Number of rows after dropping NaNs:", len(df))
df.head(5)

Number of rows after dropping NaNs: 3720


,d,b,t,r_out,n_r,n,mxx,myy,vx,vy,mzz,area,ixx,iyy,ixy,g_eff,utilization,section_param_id,section_type
0,292.940338,184.123599,16.722934,17.628862,4,-688877.841108,-1.312324e+07,2.412614e+06,-115170.575087,13682.531592,1.153415e+07,14527.235375,3.200690e+13,1.520191e+13,-0.265625,76923.076923,0.322426,0,rectangular_hollow_section
1,292.940338,184.123599,16.722934,17.628862,4,354570.903962,-2.555486e+07,-5.432602e+06,41320.270385,152295.668465,-2.124411e+07,14527.235375,3.200690e+13,1.520191e+13,-0.265625,76923.076923,0.352120,0,rectangular_hollow_section
2,292.940338,184.123599,16.722934,17.628862,4,-487911.441850,-6.665061e+06,2.604899e+07,-240323.071707,238398.799619,-1.857385e+07,14527.235375,3.200690e+13,1.520191e+13,-0.265625,76923.076923,0.762931,0,rectangular_hollow_section
3,292.940338,184.123599,16.722934,17.628862,4,-48213.704668,2.253705e+07,1.034981e+07,-233498.377101,-168306.320838,1.210748e+07,14527.235375,3.200690e+13,1.520191e+13,-0.265625,76923.076923,0.647851,0,rectangular_hollow_section
4,292.940338,184.123599,16.722934,17.628862,4,-957990.414679,-5.910821e+06,9.593818e+06,17546.913768,-174641.502762,1.074127e+07,14527.235375,3.200690e+13,1.520191e+13,-0.265625,76923.076923,0.274913,0,rectangular_hollow_section


In [7]:
X = df[predictor_columns]
y = df[target_columns]
X.shape, y.shape

((3720, 10), (3720, 1))

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42, stratify=df["section_param_id"]
)
print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

Training data shape: (2492, 10)
Testing data shape: (1228, 10)


## Train models

In [9]:
mlflow.set_tracking_uri(mlflow_tracking_uri)
experiment = mlflow.set_experiment(mlflow_experiment_name)
mlflow.sklearn.autolog(log_models=False)

2025/11/12 22:23:52 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2025/11/12 22:23:52 INFO mlflow.store.db.utils: Updating database tables
2025-11-12 22:23:52 INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
2025-11-12 22:23:52 INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
2025-11-12 22:23:52 INFO  [alembic.runtime.migration] Running upgrade  -> 451aebb31d03, add metric step
2025-11-12 22:23:52 INFO  [alembic.runtime.migration] Running upgrade 451aebb31d03 -> 90e64c465722, migrate user column to tags
2025-11-12 22:23:52 INFO  [alembic.runtime.migration] Running upgrade 90e64c465722 -> 181f10493468, allow nulls for metric values
2025-11-12 22:23:52 INFO  [alembic.runtime.migration] Running upgrade 181f10493468 -> df50e92ffc5e, Add Experiment Tags Table
2025-11-12 22:23:52 INFO  [alembic.runtime.migration] Running upgrade df50e92ffc5e -> 7ac759974ad8, Update run tags with larger limit
2025-11-12 22:23:52 INFO  [alembic.runtime.mig

In [ ]:
def train_model(
    model: BaseEstimator, 
    params_grid: Optional[dict] = None, 
    model_name: Optional[str] = None
) -> BaseEstimator:
    run = mlflow.start_run(run_name=model_name or model.__class__.__name__)  # --- start main run
    run_id = run.info.run_id
    try:
        # ----- tags & baseline params
        mlflow.set_tag("model_class", model.__class__.__name__)
        mlflow.set_tag("model_name", model_name)
        mlflow.set_tag("stage", "baseline" if not params_grid else "baseline+search")
        mlflow.set_tag("task", task)
        mlflow.set_tag("section_type", section_type)
        mlflow.set_tag("library", "sklearn")

        # ----- fit on train
        model.fit(X_train, y_train.values.ravel())
        
        # define cross-validation strategy and log parameters
        kf_params = {"n_splits": 6, "random_state": 42, "shuffle": True}
        kf = KFold(**kf_params)
        logged_kf_params = {f"kf__{k}": v for k, v in kf_params.items()}
        mlflow.log_params(logged_kf_params)

        # ----- hyperparameter search (optional)
        if params_grid:
            mlflow.log_dict(params_grid, "param_grid.json")
            mlflow.start_run(run_name=f"{model_name} - RandomizedSearchCV", nested=True)  #-- start nested run
            try:
                cv = RandomizedSearchCV(
                    estimator=model,
                    param_distributions=params_grid,
                    cv=kf,
                    n_iter=10,
                    n_jobs=-1,
                    random_state=42,
                    refit=True,
                )
                cv.fit(X_train, y_train.values.ravel())
                model = cv.best_estimator_
            finally:
                mlflow.end_run()  # --- end nested run
        
        # log the final model
        mlflow.sklearn.log_model(model, name=model_name, input_example=X_train.iloc[:5])
        
        # log cv scores
        cv_scores = cross_val_score(model, X_train, y_train.values.ravel(), cv=kf)
        mlflow.log_metric("train_cv_score_mean", float(np.mean(cv_scores)))
        mlflow.log_metric("train_cv_score_std", float(np.std(cv_scores)))
        
        # log regression metrics on test set
        y_true = y_test.values.ravel()
        y_pred = model.predict(X_test)
        mae = mean_absolute_error(y_true, y_pred)
        mse = mean_squared_error(y_true, y_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_true, y_pred)
        mape = mean_absolute_percentage_error(y_true, y_pred)
        medae = median_absolute_error(y_true, y_pred)
        metrics = {
            "test_mae": mae,
            "test_mse": mse,
            "test_rmse": rmse,
            "test_r2": r2,
            "test_mape": mape,
            "test_medae": medae,
        }
        mlflow.log_metrics(metrics)
        
        # log canonical score - the higher the better
        canonical_score = canonical_regression_score(y_true, y_pred)
        mlflow.log_metric(CANONICAL_SCORE_NAME, canonical_score)

        return run_id, model

    finally:
        mlflow.end_run()  # --- end main run
        print("MLflow run ended.")

### Baseline model

In [11]:
model_name = "Linear Regression Baseline"
steps = [
    ("scaling", StandardScaler()),
    ("regression", LinearRegression())
]
pipeline = Pipeline(steps)

run_id, model = train_model(pipeline, model_name=model_name)

MLflow run ended.


### Tuned models

In [12]:
model_name = "Ridge Regression"
steps = [
    ("scaling", StandardScaler()),
    ("regression", Ridge(alpha=0.1))
]
pipeline = Pipeline(steps)
params_grid = {"regression__alpha": np.arange(0.0001, 1, 10)}

run_id, model = train_model(pipeline, params_grid=params_grid, model_name=model_name)

/Users/baloghbence/Library/Caches/pypoetry/virtualenvs/sigmaepsilon-solid-fourier-2jj81J8s-py3.12/lib/python3.12/site-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 1 is smaller than n_iter=10. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
2025/11/12 22:24:00 INFO mlflow.sklearn.utils: Logging the 5 best runs, no runs will be omitted.
2025/11/12 22:24:02 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during sklearn autologging: The following failures occurred while performing one or more logging operations: [MlflowException('Failed to perform one or more operations on the run with ID 95c9a7f232024633bb26bd1817ce549d. Failed operations: [MlflowException(\'Changing param values is not allowed. Params were already logged=\\\'[{\\\'key\\\': \\\'steps\\\', \\\'old_value\\\': "[(\\\'scaling\\\', StandardScaler()), (\\\'regression\\\', Ridge(alpha=0.1))]", \\\'new_value\\\': "[(\\\'scaling\\

MLflow run ended.


In [13]:
model_name = "Lasso Regression"
steps = [
    ("scaling", StandardScaler()),
    ("regression", Lasso(alpha=0.1))
]
pipeline = Pipeline(steps)
params_grid = {"regression__alpha": np.arange(0.0001, 1, 10)}

run_id, model = train_model(pipeline, params_grid=params_grid, model_name=model_name)

/Users/baloghbence/Library/Caches/pypoetry/virtualenvs/sigmaepsilon-solid-fourier-2jj81J8s-py3.12/lib/python3.12/site-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 1 is smaller than n_iter=10. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
2025/11/12 22:24:04 INFO mlflow.sklearn.utils: Logging the 5 best runs, no runs will be omitted.
2025/11/12 22:24:06 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during sklearn autologging: The following failures occurred while performing one or more logging operations: [MlflowException('Failed to perform one or more operations on the run with ID 0773fd4bd50f44568eee4b9bd8980c5e. Failed operations: [MlflowException(\'Changing param values is not allowed. Params were already logged=\\\'[{\\\'key\\\': \\\'steps\\\', \\\'old_value\\\': "[(\\\'scaling\\\', StandardScaler()), (\\\'regression\\\', Lasso(alpha=0.1))]", \\\'new_value\\\': "[(\\\'scaling\\

MLflow run ended.


In [14]:
model_name = "Lasso Regression Poly3"
steps = [
    ("feature_eng", PolynomialFeatures(3, include_bias=False)),
    ("scaling", StandardScaler()),
    ("regression", Lasso(alpha=0.1))
]
pipeline = Pipeline(steps)
params_grid = {"regression__alpha": np.arange(0.0001, 1, 10)}

run_id, model = train_model(pipeline, params_grid=params_grid, model_name=model_name)

/Users/baloghbence/Library/Caches/pypoetry/virtualenvs/sigmaepsilon-solid-fourier-2jj81J8s-py3.12/lib/python3.12/site-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 1 is smaller than n_iter=10. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
/Users/baloghbence/Library/Caches/pypoetry/virtualenvs/sigmaepsilon-solid-fourier-2jj81J8s-py3.12/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.811e+02, tolerance: 6.137e-01
  model = cd_fast.enet_coordinate_descent(
/Users/baloghbence/Library/Caches/pypoetry/virtualenvs/sigmaepsilon-solid-fourier-2jj81J8s-py3.12/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to i

MLflow run ended.


In [15]:
model_name = "Ridge Regression Poly3"
steps = [
    ("feature_eng", PolynomialFeatures(3, include_bias=False)),
    ("scaling", StandardScaler()),
    ("regression", Ridge(alpha=0.1))
]
pipeline = Pipeline(steps)
params_grid = {"regression__alpha": np.arange(0.0001, 1, 10)}

run_id, model = train_model(pipeline, params_grid=params_grid, model_name=model_name)

/Users/baloghbence/Library/Caches/pypoetry/virtualenvs/sigmaepsilon-solid-fourier-2jj81J8s-py3.12/lib/python3.12/site-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 1 is smaller than n_iter=10. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
2025/11/12 22:24:16 INFO mlflow.sklearn.utils: Logging the 5 best runs, no runs will be omitted.
2025/11/12 22:24:18 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during sklearn autologging: The following failures occurred while performing one or more logging operations: [MlflowException('Failed to perform one or more operations on the run with ID 1a9f5d3e5f4c4ee78c5055c1978f673a. Failed operations: [MlflowException(\'Changing param values is not allowed. Params were already logged=\\\'[{\\\'key\\\': \\\'steps\\\', \\\'old_value\\\': "[(\\\'feature_eng\\\', PolynomialFeatures(degree=3, include_bias=False)), (\\\'scaling\\\', StandardScaler()), (\\\

MLflow run ended.
